# 🔍 Module 5: Unified Search Engine (Body + Face)

**Role in the project:** Query layer of the Re-ID system. Given a body or face image it returns the matching global identity together with the trajectory and sightings stored in Pinecone/MongoDB.

**Architecture:** α-FPS Prototypes (Body ReID) + Direct Face Search (ArcFace) + Pinecone + MongoDB

**Pipeline:**
1. Load `identity_buffer.pkl` produced by **Module 4**
2. Run α-FPS to extract K=3 body prototypes per person → upsert to `supermarket-reid` Pinecone index
3. Face embeddings already live in `supermarket-faces` (uploaded by Module 4)
4. Search by **body crop** or **face crop** → fetch trajectory & sightings from MongoDB

**Inputs:** `identity_buffer.pkl` from Module 4, plus a query image (URL or upload)
**Outputs:** ranked identity matches + metadata; the search functions are also reused by the Module 6 dashboard
**Depends on:** Module 4 outputs (Pinecone indices + `identity_buffer.pkl`)


## 1. Environment Setup & Credentials

In [ ]:
%%capture
!pip install ultralytics boxmot opencv-python scipy torch torchvision
!pip install pinecone pymongo matplotlib
!pip install insightface onnxruntime-gpu

print("Dependencies Ready.")

In [ ]:
import os, cv2, torch, pickle, time, shutil
import numpy as np
import urllib.request
import matplotlib.pyplot as plt
import torch.nn.functional as F
from pathlib import Path
from pinecone import Pinecone
from pymongo import MongoClient
from kaggle_secrets import UserSecretsClient
from tqdm.auto import tqdm

# ═══════════════════════════════════════════════════════════════
# CREDENTIALS
# ═══════════════════════════════════════════════════════════════
try:
    secrets = UserSecretsClient()
    PINECONE_API_KEY = secrets.get_secret("PINECONE_API_KEY")
    MONGO_URI        = secrets.get_secret("MONGO_URI")
except Exception:
    PINECONE_API_KEY = "YOUR_PINECONE_KEY"
    MONGO_URI        = "YOUR_MONGO_URI"

# ═══════════════════════════════════════════════════════════════
# PINECONE INDICES
# ═══════════════════════════════════════════════════════════════
PINECONE_INDEX_NAME      = "supermarket-reid"   # 1280D cosine – body ReID prototypes
PINECONE_FACE_INDEX_NAME = "supermarket-faces"  # 512D cosine  – face embeddings (from Module 4)

# ═══════════════════════════════════════════════════════════════
# PATHS
# ═══════════════════════════════════════════════════════════════
BUFFER_PATH = Path('/kaggle/input/datasets/abderrahimzineddine/module2-embeddings/identity_buffer(1).pkl')

print("Configuration & Credentials Ready.")

## 2. Cloud Connections & Models

In [ ]:
import boxmot, pkgutil, inspect
from insightface.app import FaceAnalysis

# — Cloud databases —
print("Connecting to Pinecone...")
pc = Pinecone(api_key=PINECONE_API_KEY)
index      = pc.Index(PINECONE_INDEX_NAME)
face_index = pc.Index(PINECONE_FACE_INDEX_NAME)

print("Connecting to MongoDB Atlas...")
mongo_client = MongoClient(MONGO_URI)
db = mongo_client["supermarket_db"]
persons_col = db["persons"]

# — Body ReID model (same CLIP-ReID from Module 2/4) —
REID_DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

ReID = None
for importer, modname, ispkg in pkgutil.walk_packages(boxmot.__path__, boxmot.__name__ + '.'):
    try:
        mod = __import__(modname, fromlist="dummy")
        for name, obj in inspect.getmembers(mod):
            if inspect.isclass(obj) and name in ['ReID', 'ReIDDetectMultiBackend']:
                ReID = obj; break
    except Exception: continue
    if ReID is not None: break

print("Loading CLIP-Market1501 (Body ReID)...")
reid_model = ReID(weights='clip_market1501.pt', device=REID_DEVICE, half=False)

def embed_body(img_bgr):
    """Embed a body crop → 512-d normalized tensor."""
    with torch.no_grad():
        emb = reid_model([img_bgr])
        if isinstance(emb, np.ndarray): emb = torch.tensor(emb)
        return F.normalize(emb, p=2, dim=1).to(REID_DEVICE)

# — Face model (InsightFace/ArcFace) —
print("Loading InsightFace (ArcFace)...")
face_app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
face_app.prepare(ctx_id=0, det_size=(640, 640))

def embed_face(img_bgr):
    """Detect face in image → return (512-d numpy embedding, det_score) or (None, 0)."""
    faces = face_app.get(img_bgr)
    if not faces:
        return None, 0.0
    best = max(faces, key=lambda f: f.det_score)
    return best.normed_embedding, float(best.det_score)


def embed_face_crop(img_bgr):
    """Embed a PRE-CROPPED face image (already tightly cropped).
    Pads the crop into a larger canvas so InsightFace's detector can find it."""
    h, w = img_bgr.shape[:2]
    # Pad into a 3x larger canvas with gray border
    pad = max(h, w)
    canvas = np.full((h + 2*pad, w + 2*pad, 3), 128, dtype=np.uint8)
    canvas[pad:pad+h, pad:pad+w] = img_bgr
    return embed_face(canvas)

print("All models loaded.")

## 3. α-FPS Prototype Builder (Body Index)
Calculates K=3 optimal prototypes per person based on quality AND geometric diversity.
Run this **once** after Module 4 to populate the `supermarket-reid` Pinecone index.

In [ ]:
if not BUFFER_PATH.exists():
    print("identity_buffer.pkl not found!")
else:
    # Try to find a local copy of saved identities for images
    local_db = Path('/kaggle/input/datasets/abderrahimzineddine/module2-saved-identities')
    if not local_db.exists():
        local_db = Path('/kaggle/working/module2_database/saved_identities')

    with open(BUFFER_PATH, 'rb') as f:
        saved_data = pickle.load(f)

    print(f"\n🖼️ Visualizing K=3 Prototypes for {len(saved_data)} Identities...")

    for gid, data in saved_data.items():
        embs = torch.tensor(np.array(data["embeddings"])).to(REID_DEVICE)
        quality_scores = torch.tensor(data["det_scores"]).to(REID_DEVICE) * torch.tensor(data["ratios"]).to(REID_DEVICE)

        best_indices = alpha_fps(embs, quality_scores, K=3, alpha=0.5)
        filepaths = data.get("filepaths", [])

        if not filepaths:
            continue

        # Clamp indices to valid filepath range
        valid_indices = [idx for idx in best_indices if idx < len(filepaths)]
        if not valid_indices:
            continue

        fig, axes = plt.subplots(1, len(valid_indices), figsize=(12, 4))
        if len(valid_indices) == 1:
            axes = [axes]

        fig.suptitle(f"Global ID {gid} | Body Prototypes", fontsize=14, fontweight="bold")

        for i, idx in enumerate(valid_indices):
            raw_path = Path(filepaths[idx])

            # Strategy 1: Try local_db remapping
            actual = local_db / raw_path.parent.name / raw_path.name
            img_bgr = None

            if actual.exists():
                img_bgr = cv2.imread(str(actual))
            elif raw_path.exists():
                img_bgr = cv2.imread(str(raw_path))
            else:
                # Strategy 2: Try MongoDB best_crop_url for this GID
                person_doc = persons_col.find_one({"global_id": int(gid)}, {"_id": 0})
                if person_doc and person_doc.get("sightings"):
                    for s in person_doc["sightings"]:
                        crop_url = s.get("best_crop_url", "")
                        if crop_url:
                            try:
                                req = urllib.request.Request(crop_url, headers={'User-Agent': 'Mozilla/5.0'})
                                with urllib.request.urlopen(req) as resp:
                                    arr = np.asarray(bytearray(resp.read()), dtype=np.uint8)
                                    img_bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
                            except: pass
                            if img_bgr is not None:
                                break

            if img_bgr is not None:
                img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
                axes[i].imshow(img_rgb)
            else:
                axes[i].imshow(np.zeros((256, 192, 3), dtype=np.uint8))

            axes[i].set_title(f"Proto {i+1}\nIdx: {idx} | Q: {quality_scores[idx]:.2f}")
            axes[i].axis("off")

        plt.tight_layout()
        plt.show()

## 4. Search Functions

### 4a. Search by Body Crop
Queries `supermarket-reid` with α-FPS prototypes, applies softmax ambiguity check.

In [ ]:
def search_person(query_img_bgr, top_k=3, th_emb=0.75, tau=1.5):
    """Body search: embed → query supermarket-reid → softmax ambiguity → MongoDB."""
    query_emb = embed_body(query_img_bgr)
    vector_list = query_emb.cpu().numpy().tolist()[0]

    response = index.query(vector=vector_list, top_k=10, include_metadata=True)

    if not response.matches or response.matches[0].score < th_emb:
        return {"status": "UNKNOWN", "reason": "Below absolute similarity threshold."}

    # Group by global_id, take MAX score per ID
    id_scores = {}
    for match in response.matches:
        gid = match.metadata["global_id"]
        if gid not in id_scores:
            id_scores[gid] = match.score

    sorted_ids = sorted(id_scores.items(), key=lambda x: x[1], reverse=True)
    best_gid, best_score = sorted_ids[0]

    # Softmax ambiguity check
    if len(sorted_ids) > 1:
        s1, s2 = sorted_ids[0][1], sorted_ids[1][1]
        probs = F.softmax(torch.tensor([s1, s2]) * 15.0, dim=0)
        ratio = probs[0].item() / (probs[1].item() + 1e-9)
        if ratio < tau:
            return {
                "status": "AMBIGUOUS",
                "reason": f"Confused between GID {best_gid} and GID {sorted_ids[1][0]}. Ratio {ratio:.2f} < {tau}",
                "best_guess_gid": best_gid
            }

    person_doc = persons_col.find_one({"global_id": int(best_gid)}, {"_id": 0})
    if not person_doc:
        return {"status": "ERROR", "reason": "Match in Pinecone but no MongoDB document."}

    return {"status": "MATCH", "global_id": best_gid, "score": best_score, "mongodb_data": person_doc}

### 4b. Search by Face
Queries `supermarket-faces` directly — no prototypes needed, face embeddings are ground truth.

In [ ]:
FACE_SEARCH_THRESHOLD = 0.55  # minimum cosine similarity for a face match

def search_face(query_img_bgr, th_face=FACE_SEARCH_THRESHOLD):
    """Face search: detect face → embed → query supermarket-faces → MongoDB."""
    face_emb, det_score = embed_face(query_img_bgr)

    if face_emb is None:
        return {"status": "NO_FACE", "reason": "No face detected in the query image."}

    response = face_index.query(vector=face_emb.tolist(), top_k=5, include_metadata=True)

    if not response.matches or response.matches[0].score < th_face:
        return {"status": "UNKNOWN", "reason": f"No face match above threshold ({th_face})."}

    best_match = response.matches[0]
    best_gid = int(best_match.metadata["global_id"])
    best_score = best_match.score

    person_doc = persons_col.find_one({"global_id": best_gid}, {"_id": 0})
    if not person_doc:
        return {"status": "ERROR", "reason": "Face match in Pinecone but no MongoDB document."}

    return {
        "status": "MATCH",
        "global_id": best_gid,
        "score": best_score,
        "det_score": det_score,
        "mongodb_data": person_doc
    }

## 5. Interactive Search Tests

### 5a. Search by Body Image (URL)

In [ ]:
def fetch_image_from_url(url, as_rgb=False):
    """Download an image from any URL."""
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response:
            arr = np.asarray(bytearray(response.read()), dtype=np.uint8)
            img_bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
            if as_rgb:
                return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            return img_bgr
    except Exception as e:
        print(f"Failed to load URL: {e}")
        return np.zeros((256, 192, 3), dtype=np.uint8)


def visualize_search_result(query_img_bgr, result, title_prefix=""):
    """Plot query vs match side by side."""
    plt.figure(figsize=(10, 5))

    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(query_img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(f"{title_prefix}Query Image")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    if result["status"] == "MATCH":
        mongo_data = result.get("mongodb_data", {})
        sightings = mongo_data.get("sightings", [])
        faces = mongo_data.get("faces", [])

        # Try: 1) sighting crop URL  2) face URL  3) black
        match_url = None
        for s in sightings:
            if s.get("best_crop_url"):
                match_url = s["best_crop_url"]
                break
        if not match_url:
            for f in faces:
                if f.get("url"):
                    match_url = f["url"]
                    break
        if not match_url:
            match_url = result.get("match_face_url", "")

        if match_url:
            match_img_rgb = fetch_image_from_url(match_url, as_rgb=True)
            plt.imshow(match_img_rgb)
        else:
            plt.imshow(np.zeros((256, 192, 3), dtype=np.uint8))

        src_label = "face" if (not sightings and faces) else "body"
        title = f"MATCH: Global ID {result['global_id']}\n"
        title += f"Score: {result['score']:.3f}\n"
        title += f"Sightings: {len(sightings)} | Faces: {len(faces)} ({src_label})"
        plt.title(title, color="green", fontweight="bold")
    else:
        plt.imshow(np.zeros((256, 192, 3), dtype=np.uint8))
        plt.title(f"{result['status']}\n{result['reason']}", color="red")

    plt.axis("off")
    plt.tight_layout()
    plt.show()


# — Body search test —
TEST_BODY_URL = "https://images.unsplash.com/photo-1500648767791-00dcc994a43e"  # <-- PASTE BODY CROP URL

print("\n🔍 Searching by BODY...")
query_body = fetch_image_from_url(TEST_BODY_URL, as_rgb=False)
if query_body is not None and query_body.any():
    result_body = search_person(query_body)
    visualize_search_result(query_body, result_body, title_prefix="[Body] ")
else:
    print("Invalid URL or image could not be loaded.")

### 5b. Search by Face Image (URL)

In [ ]:
# — Face search test —
TEST_FACE_URL = "https://ik.imagekit.io/ZenoReid/supermarket_reid/global_id_027/face_g027_t1_1778081580_LBMdsPsaeQ.jpg?updatedAt=1778081581310"  # <-- PASTE URL

print("\n🔍 Searching by FACE...")
query_face = fetch_image_from_url(TEST_FACE_URL, as_rgb=False)
if query_face is not None and query_face.any():
    # Use padded embedding for pre-cropped face chips (e.g. from ImageKit)
    face_emb, det_score = embed_face_crop(query_face)

    if face_emb is None:
        # Fallback: try raw detection (works for full-scene images)
        face_emb, det_score = embed_face(query_face)

    if face_emb is None:
        print("❌ No face detected even after padding.")
    else:
        response = face_index.query(vector=face_emb.tolist(), top_k=5, include_metadata=True)
        if not response.matches or response.matches[0].score < FACE_SEARCH_THRESHOLD:
            result_face = {"status": "UNKNOWN", "reason": f"No match above threshold ({FACE_SEARCH_THRESHOLD})."}
        else:
            best = response.matches[0]
            best_gid = int(best.metadata["global_id"])
            person_doc = persons_col.find_one({"global_id": best_gid}, {"_id": 0})
            result_face = {
                "status": "MATCH", "global_id": best_gid,
                "score": best.score, "mongodb_data": person_doc or {},
                "match_face_url": best.metadata.get("face_url", "")
            }
        visualize_search_result(query_face, result_face, title_prefix="[Face] ")
else:
    print("Invalid URL or image could not be loaded.")

## 6. Brute Force Stress Tests

### 6a. Body Brute Force Test
Iterates through all saved body crops and checks if the search engine returns the correct ID.

In [ ]:
local_db = Path('/kaggle/input/datasets/abderrahimzineddine/module2-saved-identities')
if not local_db.exists():
    local_db = Path('/kaggle/working/module2_database/saved_identities')

eval_out_dir = Path('/kaggle/working/eval_body_failures')
failed_dir = eval_out_dir / 'failed_misclassified'
rejected_dir = eval_out_dir / 'rejected_unknown'

if eval_out_dir.exists():
    shutil.rmtree(eval_out_dir)
failed_dir.mkdir(parents=True, exist_ok=True)
rejected_dir.mkdir(parents=True, exist_ok=True)

all_images = list(local_db.rglob("*.jpg"))

if not all_images:
    print(f"❌ No images found in '{local_db}'.")
else:
    print(f"\n🚀 Body Brute Force Test on {len(all_images)} crops...")

    total, passed, failed_wrong, rej_ambig, rej_unknown = 0, 0, 0, 0, 0
    search_times, sim_scores = [], []

    for img_path in tqdm(all_images, desc="Body Eval"):
        try:
            true_gid = int(img_path.parent.name.split('_')[-1])
        except: continue

        query = cv2.imread(str(img_path))
        if query is None: continue
        total += 1

        t0 = time.perf_counter()
        result = search_person(query)
        search_times.append((time.perf_counter() - t0) * 1000)

        if result["status"] == "MATCH":
            pred_gid = int(result["global_id"])
            sim_scores.append(result["score"])
            if pred_gid == true_gid:
                passed += 1
            else:
                failed_wrong += 1
                cv2.imwrite(str(failed_dir / f"true{true_gid:03d}_pred{pred_gid:03d}_{result['score']:.2f}.jpg"), query)
        elif result["status"] == "AMBIGUOUS":
            rej_ambig += 1
            cv2.imwrite(str(rejected_dir / f"true{true_gid:03d}_AMBIGUOUS.jpg"), query)
        elif result["status"] == "UNKNOWN":
            rej_unknown += 1
            cv2.imwrite(str(rejected_dir / f"true{true_gid:03d}_UNKNOWN.jpg"), query)

    acc = (passed / total * 100) if total > 0 else 0
    print("\n" + "="*50)
    print(" 📊 BODY BRUTE FORCE RESULTS")
    print("="*50)
    print(f"Total Tested       : {total}")
    print(f"✅ Correct           : {passed} ({acc:.2f}%)")
    print(f"❌ Wrong ID          : {failed_wrong}")
    print(f"⚠️ Ambiguous         : {rej_ambig}")
    print(f"⚠️ Unknown           : {rej_unknown}")
    if search_times:
        print(f"Speed (ms)         : Avg {np.mean(search_times):.1f} | Min {np.min(search_times):.1f} | Max {np.max(search_times):.1f}")
    if sim_scores:
        print(f"Similarity         : Avg {np.mean(sim_scores):.3f} | Min {np.min(sim_scores):.3f} | Max {np.max(sim_scores):.3f}")
    print("="*50)

    if failed_wrong > 0 or rej_ambig > 0 or rej_unknown > 0:
        shutil.make_archive('/kaggle/working/eval_body_failures', 'zip', str(eval_out_dir))
        print("📦 Download 'eval_body_failures.zip' for visual debugging.")

### 6b. Face Brute Force Test
For each person in MongoDB that has face URLs, download them and test the face search engine.

In [ ]:
print("\n🚀 Face Brute Force Test...")

all_persons = list(persons_col.find({}, {"_id": 0, "global_id": 1, "faces": 1}))
persons_with_faces = [p for p in all_persons if p.get("faces")]

if not persons_with_faces:
    print("❌ No persons with face data found in MongoDB.")
else:
    total_f, passed_f, failed_f, no_face_f, unknown_f = 0, 0, 0, 0, 0
    face_times, face_scores = [], []

    for person in tqdm(persons_with_faces, desc="Face Eval"):
        true_gid = person["global_id"]
        for face_entry in person.get("faces", []):
            face_url = face_entry.get("url", "")
            if not face_url:
                continue

            face_img = fetch_image_from_url(face_url, as_rgb=False)
            if face_img is None or not face_img.any():
                continue

            total_f += 1
            t0 = time.perf_counter()
            # Use padded embedding since ImageKit stores pre-cropped face chips
            face_emb, det_score = embed_face_crop(face_img)
            if face_emb is None:
                no_face_f += 1
                continue
            response = face_index.query(vector=face_emb.tolist(), top_k=5, include_metadata=True)
            if not response.matches or response.matches[0].score < FACE_SEARCH_THRESHOLD:
                result = {"status": "UNKNOWN"}
            else:
                best_match = response.matches[0]
                result = {"status": "MATCH", "global_id": int(best_match.metadata["global_id"]),
                          "score": best_match.score}
            face_times.append((time.perf_counter() - t0) * 1000)

            if result["status"] == "MATCH":
                pred_gid = int(result["global_id"])
                face_scores.append(result["score"])
                if pred_gid == true_gid:
                    passed_f += 1
                else:
                    failed_f += 1
            else:
                unknown_f += 1

    acc_f = (passed_f / total_f * 100) if total_f > 0 else 0
    print("\n" + "="*50)
    print(" 📊 FACE BRUTE FORCE RESULTS")
    print("="*50)
    print(f"Total Faces Tested : {total_f}")
    print(f"✅ Correct           : {passed_f} ({acc_f:.2f}%)")
    print(f"❌ Wrong ID          : {failed_f}")
    print(f"⚠️ No Face Detected  : {no_face_f}")
    print(f"⚠️ Unknown           : {unknown_f}")
    if face_times:
        print(f"Speed (ms)         : Avg {np.mean(face_times):.1f} | Min {np.min(face_times):.1f} | Max {np.max(face_times):.1f}")
    if face_scores:
        print(f"Face Similarity    : Avg {np.mean(face_scores):.3f} | Min {np.min(face_scores):.3f} | Max {np.max(face_scores):.3f}")
    print("="*50)

## 7. Visualize Body Prototypes
Re-runs α-FPS on the pickle to find the winning crop indices, then displays the images.

In [ ]:
if not BUFFER_PATH.exists():
    print("identity_buffer.pkl not found!")
else:
    local_db = Path('/kaggle/input/datasets/abderrahimzineddine/module2-saved-identities')
    if not local_db.exists():
        local_db = Path('/kaggle/working/module2_database/saved_identities')

    with open(BUFFER_PATH, 'rb') as f:
        saved_data = pickle.load(f)

    print(f"\n🖼️ Visualizing K=3 Prototypes for {len(saved_data)} Identities...")

    for gid, data in saved_data.items():
        embs = torch.tensor(np.array(data["embeddings"])).to(REID_DEVICE)
        quality_scores = torch.tensor(data["det_scores"]).to(REID_DEVICE) * torch.tensor(data["ratios"]).to(REID_DEVICE)

        best_indices = alpha_fps(embs, quality_scores, K=3, alpha=0.5)
        filepaths = data.get("filepaths", [])

        if not filepaths:
            continue

        # Clamp indices to valid range (handles desynced filepaths from hydration)
        valid_indices = [idx for idx in best_indices if idx < len(filepaths)]
        if not valid_indices:
            continue

        fig, axes = plt.subplots(1, len(valid_indices), figsize=(12, 4))
        if len(valid_indices) == 1:
            axes = [axes]

        fig.suptitle(f"Global ID {gid} | Body Prototypes", fontsize=14, fontweight="bold")

        for i, idx in enumerate(valid_indices):
            raw_path = Path(filepaths[idx])
            # Try local_db remapping for read-only datasets
            actual_img_path = local_db / raw_path.parent.name / raw_path.name
            if not actual_img_path.exists():
                actual_img_path = raw_path  # fallback to original

            if actual_img_path.exists():
                img_bgr = cv2.imread(str(actual_img_path))
                img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
                axes[i].imshow(img_rgb)
            else:
                axes[i].imshow(np.zeros((256, 192, 3), dtype=np.uint8))

            axes[i].set_title(f"Proto {i+1}\nIdx: {idx} | Q: {quality_scores[idx]:.2f}")
            axes[i].axis("off")

        plt.tight_layout()
        plt.show()